In [7]:
# 根据法条把类案进行划分章节
import re
import cn2an
import json

with open("/root/data1/liang/self-correct-retriever/data/knowledge_base/balanced_key_train_cvg.json", "r", encoding="utf-8") as f:
    text = f.readlines()

# 定义正则表达式来匹配法条信息
pattern = r"根据法条第(.+?)条"

# 定义划分范围
ranges = [
    (102, 113), (114, 139), (140, 231), (232, 262), (263, 276),
    (277, 367), (368, 381), (382, 396), (397, 419), (420, 451)
]
section_split = {"法条102-113":"危害国家安全罪.json","法条114-139":"危害公共安全罪.json","法条140-231":"破坏社会主义市场经济秩序罪.json",
                 "法条232-262":"侵犯公民人身权利、民主权力罪.json","法条263-276":"侵犯财产罪.json","法条277-367":"妨害社会管理秩序罪.json",
                 "法条368-381":"危害国防利益罪.json","法条382-396":"贪污贿赂罪.json","法条397-419":"渎职罪.json",
                 "法条420-451":"军人违反职责罪.json"}


# 初始化字典，用于保存每个范围的文本内容
text_by_range = {range_: [] for range_ in ranges}

# 使用正则表达式找到所有法条的编号，并按照范围分类保存文本
for line in text:
    line = json.loads(line)
    match = line["art"]
    if match:
        try:
            article_number = cn2an.cn2an(match)
        except:
            continue
        for start, end in ranges:
            if start <= article_number <= end:
                text_by_range[(start, end)].append(line)

# 根据范围将文本保存到不同的文件
for (start, end), lines in text_by_range.items():
    if lines:
        output_filename = f"法条{start}-{end}"
        with open("/root/data1/liang/self-correct-retriever/data/hera_knowbase/法院观点/"+section_split[output_filename], "w", encoding="utf-8") as f:
            for dic in lines:
                json.dump(dic,f,ensure_ascii=False)
                f.write("\n")            


In [8]:
# 给层级化知识库的每条知识加上层级信息
import os
import json
import re
filepath = "/root/data1/liang/self-correct-retriever/data/hera_knowbase/法院观点"
for root, dirs, files in os.walk(filepath):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = []
                index = file_path.index("/hera_knowbase/") + len("/hera_knowbase/")
                remaining_path = file_path[index:]
                path_list = remaining_path.split('/')
                path_list = [part.replace(".json","") for part in path_list if part]
                lines = f.readlines()
                for line in lines:
                    line = json.loads(line)
                    article = line["art"]
                    charge = line["char"]
                    line["path_list"] = path_list + [charge]
                    data.append(line)
            new_path = file_path.replace("hera_knowbase","hera_knowbase2")
            new_dir = os.path.dirname(new_path)
            if not os.path.exists(new_dir):
                os.makedirs(new_dir)
            with open(new_path, "w", encoding="utf-8") as f:
                for dic in data:
                    json.dump(dic,f,ensure_ascii=False)
                    f.write("\n") 


In [10]:
# 给层级化知识库的每条知识加上自身id，用于子图识别
import os
import json
# 提取案例库数据的标签
import pandas as pd
import json
import random
import re
import os

def read_json_files(folder_path):
    articles,charges,keys,facts,paths = [],[],[],[],[]
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    for line in lines:
                        line = json.loads(line)          
                        article = line["art"]
                        charge = line["char"]
                        articles.append(article)
                        charges.append(charge)
                        keys.append(line["key2"])
                        facts.append(line["value"])
                        paths.append(line["path_list"])
    return facts,keys,charges,articles,paths
facts,keys,charges,articles,paths = read_json_files("/root/data1/liang/self-correct-retriever/data/hera_knowbase2/法院观点")
know_df = pd.DataFrame()
know_df["fact"],know_df["key"],know_df["charge"],know_df["article"],know_df["path"]=facts,keys,charges,articles,paths
print(len(know_df))
know_df.head(2)

know_df["path"] = ["".join(path) for path in paths]
unique_path_classes = know_df['path'].unique()
path_id_dict = {path_class: idx for idx, path_class in enumerate(unique_path_classes)}

filepath = "/root/data1/liang/self-correct-retriever/data/hera_knowbase2/法院观点"
i=0
for root, dirs, files in os.walk(filepath):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = []
                lines = f.readlines()
                for line in lines:
                    line = json.loads(line)
                    path_value = "".join(line['path_list'])
                    node_C_id = path_id_dict[path_value]
                    # line["node_id"] = [i,i+len(know_df),len(know_df)*2+node_C_id]
                    line["node_id"] = [i,i+len(know_df),len(know_df)*2+i]
                    i+=1
                    data.append(line)
            new_path = file_path.replace("hera_knowbase2","hera_knowbase3")
            new_dir = os.path.dirname(new_path)
            if not os.path.exists(new_dir):
                os.makedirs(new_dir)
            with open(new_path, "w", encoding="utf-8") as f:
                for dic in data:
                    json.dump(dic,f,ensure_ascii=False)
                    f.write("\n") 


6124
